In [1]:
import importlib.util
import inspect
from pathlib import Path

import pandas as pd

from asterix_decoder.data_items.data_item import ItemXXX


# ── Helpers ───────────────────────────────────────────────────────────────────

def first_existing_path(candidates: list) -> Path | None:
    """Return the first path in the list that exists on disk."""
    for path in map(Path, candidates):
        if path.exists():
            return path
    return None


# ==============================================================================
# BLOQ 1 · Load binary file (CAT021 source)
# Goal: read the raw ASTERIX binary stream from disk into memory.
# ==============================================================================

BINARY_PATH = first_existing_path([
    Path("raw_data/datos_asterix_adsb.ast"),
    Path("raw_data/datos_asterix_radar.ast"),
])
if BINARY_PATH is None:
    raise FileNotFoundError(
        "No ASTERIX binary file found. Update the path candidates."
    )

raw_data: bytes = BINARY_PATH.read_bytes()

if len(raw_data) < 3:
    raise ValueError("File is too short to contain a valid ASTERIX header.")

print(f"Loaded : {BINARY_PATH}")
print(f"Size   : {len(raw_data):,} bytes")
print(f"First header → CAT={raw_data[0]} | LEN={int.from_bytes(raw_data[1:3], 'big')}")

Loaded : raw_data\datos_asterix_adsb.ast
Size   : 86,455,774 bytes
First header → CAT=21 | LEN=90


In [5]:
# ==============================================================================
# BLOQ 2 · Split binary stream into messages
# Goal: parse raw bytes into messages_df and keep only CAT021 messages.
# ==============================================================================

def split_messages(raw: bytes) -> pd.DataFrame:
    """Walk the binary stream and return a DataFrame with one row per message."""
    view = memoryview(raw)
    total = len(raw)
    rows, offset, msg_id = [], 0, 1

    while offset + 3 <= total:
        cat = view[offset]
        length = int.from_bytes(view[offset + 1:offset + 3], "big")

        if length < 3 or offset + length > total:
            raise ValueError(f"Invalid message at offset {offset}: LEN={length}")

        rows.append(
            {
                "message_id": msg_id,
                "offset": offset,
                "cat": cat,
                "length": length,
                "data_record": bytes(view[offset + 3:offset + length]),
            }
        )
        offset += length
        msg_id += 1

    return pd.DataFrame(rows)


messages_df = split_messages(raw_data)

print(f"\nMessages found (all CATs): {len(messages_df):,}")
print(messages_df["cat"].value_counts().sort_index().rename("count").rename_axis("CAT").to_string())

messages_df = messages_df[messages_df["cat"] == 21].copy().reset_index(drop=True)
if messages_df.empty:
    raise ValueError("No CAT021 messages found in the selected binary file.")

messages_df["message_id"] = range(1, len(messages_df) + 1)
print(f"\nCAT021 messages selected: {len(messages_df):,}")


Messages found (all CATs): 975,101
CAT
21    975101

CAT021 messages selected: 975,101


In [3]:
# ==============================================================================
# BLOQ 3 · Parse FSPEC → expand to one row per field (message × FRN)
# Goal: turn messages_df into fields_df by reading each message's FSPEC and
#       exploding the active FRNs. This is the skeleton of the final table.
# ==============================================================================

_ACTIVE_OFFSETS: tuple[tuple[int, ...], ...] = tuple(
    tuple(i for i in range(7) if (b >> (7 - i)) & 1)
    for b in range(256)
)
# Precompute FX bit too (least-significant bit signals continuation)
_HAS_FX: tuple[bool, ...] = tuple(bool(b & 1) for b in range(256))


def parse_fspec(data_record: bytes) -> tuple[list[int], bytes]:
    frns: list[int] = []
    frn_base: int = 1
    cursor: int = 0
    n: int = len(data_record)

    while cursor < n:
        b = data_record[cursor]
        offsets = _ACTIVE_OFFSETS[b]
        if offsets:
            base = frn_base
            frns += [base + off for off in offsets]
        cursor += 1
        frn_base += 7
        if not _HAS_FX[b]:
            break

    return frns, data_record[cursor:]


records = messages_df["data_record"].tolist()
parsed = list(map(parse_fspec, records))

messages_df = messages_df.assign(
    frns=[p[0] for p in parsed],
    data_fields=[p[1] for p in parsed],
)

print(messages_df[["message_id", "cat", "frns", "data_fields"]].head(8).to_string(index=False))

 message_id  cat                                                                                                frns                                                                                                                                                                                                                                                                                     data_fields
          1   21     [1, 2, 3, 4, 7, 11, 12, 14, 16, 17, 18, 21, 23, 24, 26, 28, 29, 30, 32, 35, 36, 38, 41, 42, 48]                   b'\x14\xdb\x01\x01\x00\x03\xc3\x01\x0f\x04\xb5(\x00\xe4\xf5\x95D\x03V\x1c \xa7\x1c \xb6\x10d1\xf3\x15\xb0\x12\x03\xde@\x7ff\x07s\x8c\xf5\x1c \xd2P\x16q\x088 \x03\xc2\x08\x04\x18\xcd\x02\xd7\xc3Y@\x07\x03\x07\x02\x02\x03\x07\x02\x02\x0f\x03\x1a\x07\xc4\x08X\x05\x19\x00'
          2   21 [1, 2, 3, 4, 7, 11, 12, 14, 16, 17, 18, 19, 21, 23, 24, 26, 28, 29, 30, 32, 35, 36, 38, 41, 42, 48] b'\x14\xdb\x01\x01\x00\x03\xd5\x01\x0e\x05\x97x\x00?\xe2\xc9L\xa2[\x1c \x

In [4]:
from asterix_decoder.data_tables.uap_tables import uap021_df


def discover_item_classes(cats: list[int], base_folder: Path = Path("asterix_decoder/data_items")) -> dict[tuple[int, str], type]:
    """Map (cat, item_id) -> class loaded dynamically from CATxxx folders."""
    class_map: dict[tuple[int, str], type] = {}

    for cat in cats:
        folder = base_folder / f"CAT{cat:03d}"
        if not folder.exists():
            continue

        for file in folder.glob("*.py"):
            if file.name == "__init__.py":
                continue

            module_name = f"dyn_cat{cat:03d}_{file.stem}"
            spec = importlib.util.spec_from_file_location(module_name, file)
            if spec is None or spec.loader is None:
                continue

            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            for _, cls in inspect.getmembers(module, inspect.isclass):
                if cls.__module__ == module_name and callable(getattr(cls, "get_item_id", None)):
                    item_id = str(cls.get_item_id()).strip()
                    class_map[(cat, item_id)] = cls

    return class_map


def build_instance(row: pd.Series, class_map: dict[tuple[int, str], type]):
    """Create one decoder instance for one UAP row."""
    cat = int(row["cat"])
    item_id = str(row["item_id"]).strip()
    item_name = str(row["item_name"])
    length_str = str(row["length_str"])

    cls = class_map.get((cat, item_id))
    if cls is None:
        return ItemXXX(item_name=item_name, length_str=length_str, item_id=item_id)

    try:
        return cls(item_name=item_name, length_str=length_str)
    except TypeError:
        return cls(item_name, length_str)


uap_df = uap021_df.copy().reset_index(drop=True)
CLASS_MAP = discover_item_classes([21])
uap_df["instance"] = uap_df.apply(lambda r: build_instance(r, CLASS_MAP), axis=1)

print(f"UAP CAT021 rows: {len(uap_df)}")
print(uap_df[["cat", "frn", "item_id", "instance"]].head(10).to_string(index=False))

UAP CAT021 rows: 44
 cat  frn  item_id                                                                    instance
  21    1 I021/010              <dyn_cat021_item_021_010.Item010 object at 0x000001CAC0EC7CD0>
  21    2 I021/040              <dyn_cat021_item_021_040.Item040 object at 0x000001CAC0EC7A50>
  21    3 I021/161 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CAC0EC7C50>
  21    4 I021/015 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CA81165B90>
  21    5 I021/071 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CA8F2F4D10>
  21    6 I021/130 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CA920A5DD0>
  21    7 I021/131              <dyn_cat021_item_021_131.Item131 object at 0x000001CAC0EC7D90>
  21    8 I021/072 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CAC0EC7E10>
  21    9 I021/150 <asterix_decoder.data_items.data_item.ItemXXX object at 0x000001CAC0EC7E90>
  21   10 I021/151 <asterix_de

In [ ]:
def decode_message(cat: int, frns: list[int], data_fields: bytes, frn_map: dict) -> dict:
    """Decode one message payload using the UAP FRN sequence for CAT021."""
    final_data: dict = {}
    cursor = 0

    for frn in frns:
        instance = frn_map.get((cat, frn))
        if instance is None:
            continue

        remaining = data_fields[cursor:]
        if not remaining:
            break

        try:
            delta_cursor, instance_data = instance.decode(remaining)
        except Exception as exc:
            final_data[f"ERR_FRN_{frn}"] = str(exc)
            break

        if isinstance(instance_data, dict):
            final_data.update(instance_data)

        cursor += int(delta_cursor)

    return final_data


frn_map = dict(zip(uap_df[["cat", "frn"]].apply(tuple, axis=1), uap_df["instance"]))

messages_df["data"] = [
    decode_message(r.cat, r.frns, r.data_fields, frn_map)
    for r in messages_df.itertuples(index=False)
]

print(messages_df[["message_id", "cat", "data"]].head(5).to_string(index=False))

In [ ]:
from asterix_decoder.data_tables.csv_table import CSV_CAT021_COLUMNS

# ==============================================================================
# BLOQ 6 · Explode decoded data into columns and export-ready table
# ==============================================================================

target_cols = [c for c in CSV_CAT021_COLUMNS if c != "CAT"]
target_cols = ["CAT"] + target_cols

rows = []
for r in messages_df.itertuples(index=False):
    data_dict = r.data if isinstance(r.data, dict) else {}
    row_out = {col: data_dict.get(col, None) for col in target_cols[1:]}
    rows.append(row_out)

final_df = pd.DataFrame(rows, columns=target_cols[1:])
final_df.insert(0, "CAT", "CAT021")

LAT_MIN, LAT_MAX = 40.9, 41.7
LON_MIN, LON_MAX = 1.5, 2.6

lat_col = next((c for c in final_df.columns if "LAT" in c.upper()), None)
lon_col = next((c for c in final_df.columns if "LON" in c.upper()), None)

if lat_col is not None and lon_col is not None:
    lat = pd.to_numeric(final_df[lat_col], errors="coerce")
    lon = pd.to_numeric(final_df[lon_col], errors="coerce")
    in_bbox = lat.isna() | (lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))
    final_df = final_df[in_bbox].reset_index(drop=True)

print(f"Rows after decode/filter: {len(final_df):,}")
print(final_df.head(10).to_string(index=False))

In [ ]:
# ==============================================================================
# EXPORT CSV
# ==============================================================================

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "decoded_messages_CAT021.csv"

final_df.to_csv(csv_path, index=False, sep=";", decimal=",", na_rep="N/A")

print(f"Category fixed   : CAT021")
print(f"Columnas finales : {len(final_df.columns)}")
print(f"Shape final_df   : {final_df.shape}")
print(f"✓ CSV  → {csv_path}")